# Transform Circuits Data

1. Read bronze `circuits` table
1. Keep only the columns required for analytics (Drop `url` column)
1. Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
1. Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)
1. Filter out rows where `circuit_id` is null (business key validation)
1. Remove duplicate records
1. Transform values of columns `circuit_name` and `locality` to Title Case
1. Write the transformed data to silver `circuits` table


#### Entity Relationship Diagram - Formula1 Schema

![Formula1 Raw Data.png](../../z-course-images/formula1-raw-data-erd.png "Formula1 Raw Data.png")

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/03.Configuracion_Ambiente

In [0]:
%run ../../../proyecto_final_formula1/PrepAmb/04.Ayuda_Silver

In [0]:
tabla_bronze = f"{catalogo}.{esquema_bronze}.circuits"
tabla_silver = f"{catalogo}.{esquema_silver}.circuitos"

#### Step 1 - Read bronze `circuits` table

In [0]:
df_circuits = spark.table(tabla_bronze)

In [0]:
display(df_circuits)

#### Step 2 - Keep only the columns required for analytics (Drop url column)

In [0]:
from pyspark.sql import functions as F

In [0]:
df_circuits_selected = df_circuits.select(
    F.col("circuitId"),
    F.col("circuitName"),
    F.col("lat"),
    F.col("long"),
    F.col("locality"),
    F.col("country"),
    F.col("ingesta_timestamp"),
    F.col("archivo_fuente")
)

#### Step 3 & 4 - Standardise Column Names
- Standardise column names using snake_case (`circuitId` → `circuit_id`, `circuitName` → `circuit_name`)
- Rename columns to make them more meaningful (`lat` → `latitude`, `long` → `longitude`)

In [0]:
df_circuits_renombre_cols = (
    df_circuits_selected 
        .withColumnsRenamed({
            "circuitId": "id_circuito",
            "circuitName": "nombre_circuito",
            "lat": "latitud",
            "long": "longitud",
            "locality": "localidad",
            "country":"pais"
        })
)

In [0]:
display(df_circuits_renombre_cols)

#### Step 5 - Filter out rows where circuit_id is null (business key validation)

In [0]:
df_circuits_validos = df_circuits_renombre_cols.filter(
    F.col("id_circuito").isNotNull()
)

In [0]:
display(df_circuits_validos)

#### Step 6 - Remove duplicate records

In [0]:
circuits_distinct_df = df_circuits_validos.dropDuplicates(["id_circuito"])

In [0]:
display(circuits_distinct_df)

#### Step 7 - Transform values of columns `circuit_name` and `locality` to Title Case

In [0]:
df_circuits_final= (
    circuits_distinct_df
        .withColumn('nombre_circuito', F.initcap(F.col("nombre_circuito")))
        .withColumn('localidad', F.initcap(F.col("localidad")))
         .withColumn("pais", convertir_dinamico_udf(F.col("pais")))
)



In [0]:
display(df_circuits_final)

#### Step 8 - Write the transformed data to silver `circuits` table

In [0]:
(
    df_circuits_final
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(tabla_silver)
)

In [0]:
display(spark.table(tabla_silver))

In [0]:
display(spark.table(tabla_silver))